## Setting things up

### Import libraries
Add directory with the CIBUSmod modules to path to be able to import

In [1]:
import sys
import os
sys.path.insert(0, 'C:\\Users/jnka0003/Git repos/CIBUSmod')

Import CIBUSmod and packages for handling data and plotting

In [2]:
import CIBUSmod as cm

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Set up scenarios

In [3]:
# Create session
session = cm.Session(
    name = 'FORMAS_sens',
    data_path = 'C:\\Users/jnka0003/Git repos/CIBUSmod/data',
    data_path_scenarios = 'scenarios',
    data_path_output = 'output',
)

# Define scenarios
session.add_scenario(
    name = 'BL',
    scenario_workbooks = None, 
    modules = 'all',
    pars = 'all',
    years = 0
)
session.add_scenario(
    name = 'MAX_CUR (fix ani)',
    scenario_workbooks = ['sng_area'], 
    modules = 'all',
    pars = 'all',
    years = 100
)
session.add_scenario(
    name = 'ALL (fix ani)',
    scenario_workbooks = ['sng_area', 'STEERS', 'CULandDRY_COWS',
                          'WIN_LAMB', 'REC_HORSES'],
    modules = 'all',
    pars = 'all',
    years = 100
)


A scenario with the name 'BL' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'MAX_CUR (fix ani)' already exists use .update_scenario() or .remove_scenario() instead.
A scenario with the name 'ALL (fix ani)' already exists use .update_scenario() or .remove_scenario() instead.


## Run scenarios (multi proc.)

In [4]:
# Import
from concurrent.futures import ProcessPoolExecutor, as_completed
from multi_proc import do_run

In [5]:
# Create list of scenarios to run
runs = [(s,y) for s,y in session.iterate('no output')]  # 
runs

[('MAX_CUR (fix ani)', '100')]

In [6]:
%%time
# Do the multi-processing
with ProcessPoolExecutor(max_workers=8) as executor:
    
    futures = {executor.submit(do_run, session, scn_year) : scn_year for scn_year in runs}

    for future in as_completed(futures):
    
        scn, year = futures[future]
           
        try:
            t = future.result()
        except Exception as ee:
            print(f'(!!!) {scn}, {year} failed with the exception: {ee}')
        else:
            m = int(t/60)
            s = int(round(t - m*60))
            print(f'{scn}, {year} finished successfully in {m}min {s}s')
            
session.cache.clear()

MAX_CUR (fix ani), 100 finished successfully in 1min 33s
CPU times: total: 0 ns
Wall time: 1min 40s
